# ETL de ventas

## Proposito de esta guia

En este notebook construiremos un flujo ETL sencillo para leer ventas desde un archivo JSON, validarlas, estandarizarlas y guardarlas en dos archivos de salida: uno con los registros validos y otro con los errores.

La idea es entender primero el proceso y despues escribir el codigo. Por eso, cada etapa esta documentada antes de la celda que la implementa. Al finalizar la clase, el codigo puede eliminarse y conservarse solamente este Markdown como una guia de consulta: el procedimiento, las decisiones y las preguntas importantes seguiran visibles.



## 0. Crear el entorno virtual con `uv`

`uv` es una herramienta rapida para gestionar proyectos y entornos virtuales de Python. Estos comandos se ejecutan en la terminal, desde la carpeta del proyecto:

```bash
uv venv env
source env/bin/activate       # Linux/macOS
# .\env\Scripts\activate   # Windows PowerShell
uv pip install jupyter
jupyter notebook
```

En este ejemplo solo usamos la biblioteca estandar de Python, por lo que no es necesario instalar paquetes adicionales.

### Objetivo de este paso

Crear un entorno aislado evita mezclar las dependencias de este ejercicio con las de otros proyectos. `uv venv env` crea la carpeta del entorno y la activacion hace que los comandos de Python utilicen ese entorno.

En una clase conviene ejecutar estos comandos una sola vez y luego concentrarse en el flujo ETL.

## 1. Que es un ETL

ETL significa **Extract, Transform, Load** (extraer, transformar y cargar):

1. **Extract:** obtener datos desde una fuente, en este caso `ventas.json`.
2. **Transform:** validar, limpiar y convertir los datos a un formato consistente.
3. **Load:** guardar el resultado en un destino para su uso posterior.

Separar estas etapas facilita probar el proceso y localizar errores. Cada funcion tendra una responsabilidad concreta y recibira datos como entrada para devolver un resultado.

### Mapa del proceso

`ventas.json` → **Extract** → datos crudos → **Validate** → validos y errores → **Transform** → datos estandarizados → **Load** → archivos JSON de salida.

Esta separacion tambien permite reemplazar una fuente o un destino sin reescribir todo el proceso.

## 2. Extract: leer el archivo JSON

Vamos a utilizar `import json` para trabajar con el formato JSON. La instruccion `with open(...)` abre el archivo y lo cierra automaticamente al terminar, incluso si ocurre un error.

### Que debe hacer `extract`

1. Recibir la ruta del archivo de entrada.
2. Abrirlo en modo lectura y con la codificacion adecuada.
3. Convertir el JSON a estructuras de Python con `json.load`.
4. Devolver el contenido sin modificarlo.

En esta etapa no limpiamos ni corregimos datos. Extraer y transformar son responsabilidades distintas: primero conservamos lo que llego y despues decidimos como tratarlo.

**Para explicar en clase:** `with open` funciona como un administrador de recursos; al salir del bloque, el archivo se cierra automaticamente.

In [2]:
import json

def extract():
    with open("ventas.json", "r") as archivo:
        content = json.load(archivo)
    
    return content


## 3. Validar los registros

La validacion comprueba que cada registro tenga los campos obligatorios y que sus valores respeten reglas basicas. No se trata de modificar los registros, sino de decidir si cada uno puede continuar al siguiente paso. La funcion debe devolver dos listas: 
1. registros validos 
2. registros con el detalle del error.

### Reglas que podemos revisar

- El `id` debe existir y tener el tipo esperado.
- `producto` y `categoria` no deben estar vacios.
- `precio` y `cantidad` deben ser numeros validos y positivos.
- `estado` debe pertenecer a un conjunto conocido, por ejemplo `pendiente`, `completada` o `cancelada`.
- `fecha` debe respetar el formato acordado, como `AAAA-MM-DD`.

Guardar el registro rechazado junto con el motivo es mejor que descartarlo silenciosamente: permite corregir la fuente y auditar el proceso.

**Pregunta para la clase:** ¿que diferencia hay entre un registro invalido y un registro duplicado? En este flujo primero se validan los datos y luego se controlan los duplicados.

In [3]:
def date_validate(date): 
    validate = date.split("-")

    if len(validate[0]) != 4 and int(validate[0]) < 2026 and int(validate[0] > 2023):
        print("Año no valido!")
        return False
    if len(validate[1]) != 2 and int(validate[0]) <= 12 and int(validate[0] > 0):
        print("Mes no valido!")
        return False
    if len(validate[1]) != 2 and int(validate[0]) <= 31 and int(validate[0] > 0):
        print("Dia no valido!")
        return False
    return True
     


In [4]:
def validate(data):
    valid = []
    errrs = []

    estados_validos = ["pendiente", "completada", "cancelada"]

    for element in data: 
        if type(element["id"]) is not int: 
            errrs.append(element)
            print("Id no valido!, guardado en errores.")
            continue
        if not element["producto"]: 
            print("Nombre del producto no valido, guardando en errores!")
            errrs.append(element)
            continue
        if not element["categoria"]: 
            print("Categoria no valida, guardando en errores!")
            errrs.append(element)
            continue
        if type(element["precio"]) is not int or element["precio"] <= 0:
            errrs.append(element)
            print("Precio no valido!")
            continue 
        if type(element["cantidad"]) is not int or element["cantidad"] <= 0:
            errrs.append(element)
            print("Cantidad no valida!") 
            continue
        if not element["estado"].lower().strip() in estados_validos: 
            errrs.append(element)
            print("Estado no valido!")
            continue
        if not date_validate(element["fecha"]):
            errrs.append(element)
            print("Fecha no valida!")
            continue
        valid.append(element)

    return valid, errrs

## 4. Transformar y eliminar duplicados

En esta etapa se estandarizan los textos (espacios y mayusculas/minusculas), se convierten los tipos necesarios y se conserva una sola aparicion de cada `id`. Los duplicados se envian a la lista de errores para no perder trazabilidad.

### Por que transformar

Dos valores pueden representar lo mismo aunque esten escritos de forma diferente: ` tecnologia `, `Tecnologia` y `TECNOLOGIA`. Quitar espacios innecesarios y unificar el uso de mayusculas permite agrupar, comparar y analizar sin resultados inconsistentes.

Para eliminar duplicados necesitamos definir una clave. En este ejemplo usamos `id`: si vuelve a aparecer, conservamos la primera aparicion y registramos la repetida como incidencia. La regla debe explicarse antes de implementarse, porque en otros contextos podria ser necesario combinar registros en vez de descartar uno.

**Antes de ejecutar:** decidir que campos se normalizan, cual identifica un duplicado y que registro se conserva.

In [5]:
def transform(data): 
    transforms = []
    for element in data: 
        new_element = {}
        new_element["id"] = element["id"]
        new_element["producto"] = element["producto"].upper().strip()
        new_element["categoria"] = element["categoria"].upper().strip()
        new_element["estado"] = element["estado"].upper().strip()
        new_element["vendedor"] = element["vendedor"].upper().strip()
        new_element["precio"] = element["precio"]
        new_element["cantidad"] = element["cantidad"]
        new_element["ciudad"] = element["ciudad"].upper().strip()
        if new_element in transforms:
            print("duplicado eliminado!")
            continue

        transforms.append(new_element)

    return transforms



## 5. Load: guardar resultados

La funcion `load` exporta cada resultado en un archivo JSON independiente. El objetivo es que los registros limpios puedan consumirse y que los errores puedan revisarse o corregirse sin perder informacion. Para ello utilizaremos el metodo `dump()`. Como parametro deberemos pasarle lo siguiente:  
* Contenido
* `ensure_ascii=False` para conserva caracteres como las tildes 
* `indent=4` deja los archivos faciles de leer.

### Resultado esperado

- `valids.json`: contiene solamente los registros que pasaron la validacion y la transformacion.
- `errores.json`: contiene los registros rechazados y el motivo del rechazo.

Separar ambos archivos evita que un dato defectuoso contamine el resultado principal y, al mismo tiempo, evita perder evidencia del problema.

In [6]:
def load(valid, errors): 
    with open("valids.json", "w") as file: 
        json.dump(
            valid,
            file,
            ensure_ascii=False,
            indent=4
        )
    with open("errores.json", "w") as file_2:
        json.dump(
            errors,
            file_2,
            ensure_ascii=False,
            indent=4
        )

In [7]:
def main():
    content = extract()
    valid, errors = validate(content)
    transforms = transform(valid)
    load(transforms, errors)
main()

Precio no valido!
Nombre del producto no valido, guardando en errores!
Precio no valido!
Cantidad no valida!
Precio no valido!
Precio no valido!
Nombre del producto no valido, guardando en errores!
Cantidad no valida!


## Ejecucion completa

El orden de ejecucion es: `extract -> validate -> transform -> load`. Si se modifica el archivo de entrada, basta con volver a ejecutar las celdas en ese orden.